# 04 — Analysis
## Classification Evaluation + Recommendations + Visualization

Evaluation of classification results (rule engine + ML fallback),
recommendations for fixing anti-patterns, and final visualizations.

Modules used: `evaluator.py`, `recommender.py`

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

from recommender import Recommender

with open(Path.cwd().parent / "output" / "intermediate_03.pkl", "rb") as f:
    data = pickle.load(f)

rule_results = data["rule_results"]
final_predictions = np.array(data["final_predictions"])
ml_predictions = data["ml_predictions"]
phi = data["phi"]
schema = data["schema"]
column_index = data["column_index"]
gt_labels = data["gt_labels"]

print(f"Loaded: {len(column_index)} columns")
print(f"Rule engine results: {len(rule_results)}")
print(f"Unique predictions: {len(set(final_predictions))}")

## 1. Classification Metrics

In [ ]:
# Full metrics (all columns)
f1_macro = f1_score(gt_labels, final_predictions, average="macro", zero_division=0.0)
f1_weighted = f1_score(gt_labels, final_predictions, average="weighted", zero_division=0.0)
accuracy = accuracy_score(gt_labels, final_predictions)

print("=== Full Metrics (all columns) ===")
print(f"  Accuracy:          {accuracy:.4f}")
print(f"  F1-macro:          {f1_macro:.4f}")
print(f"  F1-weighted:       {f1_weighted:.4f}")

# Active metrics (only non-clean detections)
active_mask = final_predictions != "clean"
active_pred = final_predictions[active_mask]
active_true = gt_labels[active_mask]

if len(active_pred) > 0:
    f1_macro_active = f1_score(active_true, active_pred, average="macro", zero_division=0.0)
    f1_weighted_active = f1_score(active_true, active_pred, average="weighted", zero_division=0.0)
    accuracy_active = accuracy_score(active_true, active_pred)
    print(f"\n=== Detection Precision (only {len(active_pred)} non-clean columns) ===")
    print(f"  Accuracy:          {accuracy_active:.4f}")
    print(f"  F1-macro:          {f1_macro_active:.4f}")
    print(f"  F1-weighted:       {f1_weighted_active:.4f}")
else:
    print("\nNo anti-patterns detected.")

print(f"\n  Unique classes:    {len(set(gt_labels))}")

In [ ]:
print("Classification Report (per class, all columns):")
print(classification_report(gt_labels, final_predictions, zero_division=0.0))

In [ ]:
n_matched = sum(1 for r in rule_results if r.predicted_label != "clean")
n_clean = sum(1 for p in final_predictions if p == "clean")

print("Rule Engine Coverage:")
print(f"  Total columns:     {len(column_index)}")
print(f"  Rule-matched:      {n_matched}")
print(f"  Clean:             {n_clean}")
print(f"  Coverage:          {n_matched / len(column_index):.1%}")

In [ ]:
uncovered_indices = [i for i, r in enumerate(rule_results) if r.predicted_label == "clean"]
if uncovered_indices:
    print(f"Columns not detected by rules: {len(uncovered_indices)} (treated as clean)")
    for idx in uncovered_indices[:10]:
        print(f"  {column_index[idx]}")
    if len(uncovered_indices) > 10:
        print(f"  ... and {len(uncovered_indices) - 10} more")
else:
    print("All columns covered by rules.")

## 2. Recommendations

In [ ]:
recommender = Recommender()
recommendations = recommender.recommend(schema)
print(recommender.print_recommendations(recommendations))

## 3. Feature Importance

In [ ]:
clf = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42)
scores = cross_val_score(clf, phi, gt_labels, cv=5, scoring="f1_macro")
print(f"Random Forest F1-macro (CV): {scores.mean():.4f} +/- {scores.std():.4f}")

clf.fit(phi, gt_labels)
importances = clf.feature_importances_
top_indices = np.argsort(importances)[-10:][::-1]
print("\nTop 10 features:")
for idx in top_indices:
    print(f"  Feature {idx}: {importances[idx]:.4f}")

## 4. Visualizations

In [ ]:
cm = confusion_matrix(gt_labels, final_predictions, labels=sorted(set(gt_labels)))
plt.figure(figsize=(12, 10))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=sorted(set(gt_labels)),
    yticklabels=sorted(set(gt_labels)),
)
plt.title("Confusion Matrix: Rule Engine")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
top_n = min(10, len(importances))
top_idx_sorted = np.argsort(importances)[-top_n:][::-1]

plt.figure(figsize=(10, 6))
plt.barh(range(top_n), importances[top_idx_sorted], color="steelblue")
plt.yticks(range(top_n), [f"Feature {i}" for i in top_idx_sorted])
plt.xlabel("Importance")
plt.title("Top Feature Importances (Random Forest)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 5. Generate Reports

In [ ]:
output_dir = Path.cwd().parent / "output" / "notebook_results"
output_dir.mkdir(exist_ok=True)

# Active metrics for report
active_mask = final_predictions != "clean"
active_pred = final_predictions[active_mask]
active_true = gt_labels[active_mask]
if len(active_pred) > 0:
    f1_macro_active = f1_score(active_true, active_pred, average="macro", zero_division=0.0)
    f1_weighted_active = f1_score(active_true, active_pred, average="weighted", zero_division=0.0)
    accuracy_active = accuracy_score(active_true, active_pred)
else:
    f1_macro_active = f1_weighted_active = accuracy_active = 0.0

report = "\n".join([
    "=" * 60, "EVALUATION REPORT", "=" * 60, "",
    f"Columns: {len(column_index)}",
    f"Classes: {len(set(gt_labels))}", "",
    "--- Full Metrics (all columns) ---",
    f"  Accuracy:    {accuracy:.4f}",
    f"  F1-macro:    {f1_macro:.4f}",
    f"  F1-weighted: {f1_weighted:.4f}", "",
    f"--- Detection Precision ({len(active_pred)} non-clean columns) ---",
    f"  Accuracy:    {accuracy_active:.4f}",
    f"  F1-macro:    {f1_macro_active:.4f}",
    f"  F1-weighted: {f1_weighted_active:.4f}", "",
    "--- Rule Engine Coverage ---",
    f"  Matched:     {n_matched} / {len(column_index)}",
    f"  Clean:       {n_clean}", "",
    "--- Random Forest (CV) ---",
    f"  F1-macro:    {scores.mean():.4f} +/- {scores.std():.4f}",
])

with open(output_dir / "evaluation_report.txt", "w") as f:
    f.write(report)

rec_text = recommender.print_recommendations(recommendations)

with open(output_dir / "recommendations.txt", "w") as f:
    f.write(rec_text)

print(f"Reports saved to {output_dir}")